In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import torch 
import gymnasium as gym
import mani_skill.envs
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv
from tqdm import tqdm
import pandas as pd
import time

import sys
sys.path.append('../scripts')
from baseline_model import BaselineModel
from animate_frames import animate_frames

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    raise RuntimeError('No GPU found')

/home/users/ckw24/CS372_Robot_Arm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Baseline Model Evaluation

### Qualitative

In [2]:

# Initialize model and environment 
batch_size = 16
env = gym.make(
    "PickCube-v1", # there are more tasks e.g. "PushCube-v1", "PegInsertionSide-v1", ...
    num_envs=batch_size,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_joint_delta_pos", 
    render_mode="rgb_array",
    robot_uids="so100"
)
env = ManiSkillVectorEnv(env, auto_reset=False, ignore_terminations=False)

model = BaselineModel(device, env)

# Run model
batch_frames = model.run_batch(render=True)[-1]

# Create animation of simulation results
ani = animate_frames(batch_frames)
display(HTML(ani.to_html5_video()))

### Success Rate

In [3]:
# Initialize model and environment 
batch_size = 128
env = gym.make(
    "PickCube-v1", # there are more tasks e.g. "PushCube-v1", "PegInsertionSide-v1", ...
    num_envs=batch_size,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_joint_delta_pos", 
    render_mode=None,
    robot_uids="so100"
)
env = ManiSkillVectorEnv(env, auto_reset=False, ignore_terminations=False)

model = BaselineModel(device, env)

# Run model
batch_successes = model.run_batch(render=False)[-2]

# Determine success rate
success_rate = (batch_successes.sum()/batch_successes.numel()).item()
print(f'Model success rate: {success_rate:.3f}')

Model success rate: 0.016
